# Phase 2 — 충전기 하드웨어 뉴스 수집

## 추가 배경

앱 레이어 분석(앱오류 35%, 연결실패 0.6%)만으로는 문제의 근본 원인을 알 수 없습니다.
EV 충전 서비스는 **소프트웨어(앱)** 와 **하드웨어(충전기기)** 두 레이어로 구성되므로,
하드웨어 레이어 분석을 추가해 원인을 분리합니다.

```
[앱 리뷰] 앱오류 35%
    ↕ 원인 분석
[하드웨어 뉴스] OCPP 호환 문제? 충전기 펌웨어 버그?
```

## 수집 대상

| 제조사 | 비고 |
|--------|------|
| Costel (코스텔) | 한국 EV 충전기 제조사, 동남아 수출 |
| Nancome (난컴) | 데이터 없음 (인지도 낮음) |
| OCPP 표준 관련 | 앱↔기기 통신 프로토콜 |

## 수집 소스 및 언어

앱 리뷰 수집 국가(VNM·THA·LAO·KOR)와 동일한 언어로 수집합니다:

| 소스 | 언어 | 대상 |
|------|------|------|
| 네이버 뉴스 API | 한국어 | 코스텔, OCPP, 동남아 충전기 |
| Google News RSS | 영어 | OCPP, EV charger failure, 동남아 |
| Google News RSS | 베트남어 | 현지 충전기 설치·오류 |
| Google News RSS | 태국어 | 태국 EV 충전기 품질 |

## DB 저장 구조

기존 `news_articles` 테이블에 `category='hardware_*'`로 구분해 저장합니다.
별도 테이블이 필요 없는 이유: `hardware_` prefix로 VOC 뉴스와 명확히 구분 가능.

In [ ]:
# !pip install requests feedparser python-dotenv psycopg2-binary pandas langdetect

In [ ]:
import sys
sys.path.append('../')

import os, requests, feedparser, time, re
import pandas as pd
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

from src.db import insert_df, fetch_df

CLIENT_ID     = os.getenv('NAVER_CLIENT_ID')
CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

def clean_html(t): return re.sub('<[^>]+>', '', t or '').strip()

def classify_hw(title, content):
    """하드웨어 뉴스 카테고리 분류"""
    text = f"{title} {content}".lower()
    if any(k in text for k in ['ocpp','protocol','compatibility','호환','tiêu chuẩn','มาตรฐาน','interoperability']):
        return 'hardware_protocol'
    if any(k in text for k in ['costel','nancome','manufacturer','nhà sản xuất','ผู้ผลิต','제조사','vendor']):
        return 'hardware_manufacturer'
    if any(k in text for k in ['failure','malfunction','broken','hỏng','เสีย','결함','고장','fault']):
        return 'hardware_failure'
    if any(k in text for k in ['install','deploy','lắp đặt','ติดตั้ง','설치','trạm','สถานี','infrastructure']):
        return 'hardware_infrastructure'
    return 'hardware_market'

def classify_hw_keyword(title, content):
    """하드웨어 키워드 카테고리 분류"""
    text = f"{title} {content}".lower()
    cats = {
        'OCPP·호환문제':  ['ocpp','protocol','compatibility','호환','tiêu chuẩn','มาตรฐาน'],
        '연결·통신오류':  ['connection error','lỗi kết nối','ปัญหาเชื่อมต่อ','disconnect'],
        '충전기결함':     ['defect','malfunction','broken','hỏng','เสีย','결함','고장','fault','failure'],
        '설치·인프라':   ['install','deploy','lắp đặt','ติดตั้ง','설치','trạm','สถานี','infrastructure'],
        '제조사동향':     ['costel','nancome','manufacturer','nhà sản xuất','제조사','vendor'],
        '시장·정책':     ['market','policy','regulation','thị trường','ตลาด','시장','정책','standard'],
    }
    for cat, kws in cats.items():
        if any(k in text for k in kws): return cat
    return '기타'

## 1. 네이버 뉴스 — 한국어 (코스텔·OCPP·동남아 충전기)

In [ ]:
def search_naver(query, max_items=100):
    url = 'https://openapi.naver.com/v1/search/news.json'
    headers = {'X-Naver-Client-Id': CLIENT_ID, 'X-Naver-Client-Secret': CLIENT_SECRET}
    results, start = [], 1
    while len(results) < max_items:
        params = {'query': query, 'display': min(100, max_items-len(results)),
                  'start': start, 'sort': 'date'}
        try:
            data = requests.get(url, headers=headers, params=params, timeout=10).json()
            if 'error' in data: break
            items = data.get('items', [])
            if not items: break
            results.extend(items)
            start += len(items)
            if len(items) < 100: break
            time.sleep(0.3)
        except: break
    return results

NAVER_QUERIES = [
    ('코스텔 전기차 충전기',           'KOR', 100),
    ('OCPP 전기차 충전 호환',          'MUL', 100),
    ('OCPP 충전기 오류 호환성',        'MUL',  50),
    ('동남아 EV 충전기 제조사',        'MUL', 100),
    ('라오스 EV 충전기 설치',          'LAO',  50),
    ('베트남 EV 충전 인프라 하드웨어', 'VNM',  50),
    ('전기차 충전기 고장 오류',        'MUL',  50),
]

naver_rows = []
for query, country, max_cnt in NAVER_QUERIES:
    print(f"🔍 '{query}'", end=' ')
    items = search_naver(query, max_cnt)
    for item in items:
        try:
            from email.utils import parsedate_to_datetime
            pub_date = parsedate_to_datetime(item.get('pubDate','')).strftime('%Y-%m-%d')
        except: pub_date = None
        title   = clean_html(item.get('title',''))
        content = clean_html(item.get('description',''))
        naver_rows.append({
            'source':         'naver_news',
            'publisher':      item.get('originallink','').split('/')[2] if item.get('originallink') else '',
            'title':          title, 'content': content,
            'url':            item.get('originallink') or item.get('link',''),
            'published_date': pub_date, 'country': country,
            'category':       classify_hw(title, content),
            'keyword_category': classify_hw_keyword(title, content),
        })
    print(f"→ {len(items)}건")
    time.sleep(0.4)

print(f"\n네이버 뉴스 수집: {len(naver_rows)}건")

## 2. Google News RSS — 영어·베트남어·태국어

In [ ]:
GOOGLE_QUERIES = [
    # 영어 (국제)
    ('OCPP protocol EV charger',                     'MUL', 'en'),
    ('OCPP 2.0 compatibility electric vehicle',       'MUL', 'en'),
    ('EV charger hardware malfunction Southeast Asia','MUL', 'en'),
    ('EV charger manufacturer Korea Southeast Asia',  'MUL', 'en'),
    ('EV charging app hardware communication failure','MUL', 'en'),
    # 베트남어 (VNM)
    ('thiết bị sạc điện EV lỗi hỏng hóc',            'VNM', 'vi'),
    ('trạm sạc xe điện lỗi kết nối',                 'VNM', 'vi'),
    ('OCPP tiêu chuẩn sạc xe điện',                  'MUL', 'vi'),
    # 태국어 (THA)
    ('สถานีชาร์จ EV เสียบกชำรุด ปัญหา',               'THA', 'th'),
    ('ปัญหาเชื่อมต่อ EV charger ไทย',                 'THA', 'th'),
    ('OCPP มาตรฐาน ชาร์จ EV ไทย',                    'MUL', 'th'),
]

google_rows = []
for q, country, lang in GOOGLE_QUERIES:
    encoded = requests.utils.quote(q)
    gl = {'vi':'VN','th':'TH','en':'US'}.get(lang,'US')
    url = f"https://news.google.com/rss/search?q={encoded}&hl={lang}&gl={gl}&ceid={gl}:{lang}"
    try:
        feed = feedparser.parse(url)
        for entry in feed.entries:
            title   = clean_html(entry.get('title',''))
            content = clean_html(entry.get('summary',''))
            pub_date = time.strftime('%Y-%m-%d', entry.published_parsed) if entry.get('published_parsed') else None
            pub_src  = entry.get('source',{})
            google_rows.append({
                'source':         'google_news',
                'publisher':      pub_src.get('title','') if isinstance(pub_src,dict) else '',
                'title':          title, 'content': content,
                'url':            entry.get('link',''),
                'published_date': pub_date, 'country': country,
                'category':       classify_hw(title, content),
                'keyword_category': classify_hw_keyword(title, content),
            })
        print(f"  [{lang.upper()}] '{q[:45]}' → {len(feed.entries)}건")
    except Exception as e:
        print(f"  ❌ {e}")
    time.sleep(0.5)

print(f"\nGoogle News 수집: {len(google_rows)}건")

## 3. 통합 & DB 저장

In [ ]:
# 통합 & 중복 제거
all_rows = naver_rows + google_rows
hw_df = pd.DataFrame(all_rows).drop_duplicates('url').reset_index(drop=True)
hw_df = hw_df[hw_df['title'].str.strip().astype(bool) & hw_df['content'].str.strip().astype(bool)]

# 기존 URL 중복 제거
existing_urls = set(fetch_df("SELECT url FROM news_articles WHERE category LIKE 'hardware%'")['url'].tolist())
hw_df = hw_df[~hw_df['url'].isin(existing_urls)].reset_index(drop=True)

print(f"신규 수집: {len(hw_df)}건")
print(hw_df['category'].value_counts().to_string())

# DB 저장
insert_df(hw_df, 'news_articles')

## 4. 전처리 — 언어 감지 + 감성 분석

In [ ]:
# ⚠️ 주의: 감성 분석은 XLM-RoBERTa 모델 로딩 필요 (첫 실행 시 다운로드)
# 대용량 처리 시 별도 스크립트 실행 권장

from langdetect import detect, LangDetectException
from transformers import pipeline
from src.db import get_connection

# 언어 감지
df = fetch_df("SELECT id, title, content FROM news_articles WHERE category LIKE 'hardware%' AND lang_detected IS NULL")
print(f"언어 감지 대상: {len(df)}건")

def detect_lang(text):
    try: return detect(str(text)[:300])
    except LangDetectException: return 'unknown'

lang_rows = [(detect_lang(f"{r['title']} {r['content']}"), int(r['id'])) for _, r in df.iterrows()]
for i in range(0, len(lang_rows), 50):
    batch = lang_rows[i:i+50]
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.executemany("UPDATE news_articles SET lang_detected=%s WHERE id=%s", batch)
        conn.commit()
print("✅ 언어 감지 완료")

# 감성 분석
sa = pipeline("sentiment-analysis",
              model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
              device=-1, truncation=True, max_length=128, use_fast=False)
LABEL_MAP = {'Positive':'positive','Negative':'negative','Neutral':'neutral'}

hw_sent = fetch_df("SELECT id, title, content FROM news_articles WHERE category LIKE 'hardware%' AND sentiment_label IS NULL")
print(f"감성 분석 대상: {len(hw_sent)}건")
for i in range(0, len(hw_sent), 64):
    batch = hw_sent.iloc[i:i+64]
    texts = [f"{r['title']} {r['content']}"[:512] for _, r in batch.iterrows()]
    results = sa(texts)
    rows = [(LABEL_MAP.get(r['label'], r['label'].lower()), round(r['score'],4), int(row['id']))
            for r, (_, row) in zip(results, batch.iterrows())]
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.executemany("UPDATE news_articles SET sentiment_label=%s, sentiment_score=%s WHERE id=%s", rows)
        conn.commit()
    print(f"  {min(i+64, len(hw_sent))}/{len(hw_sent)}건...", end='\r')
print("\n✅ 감성 분석 완료")

## 5. 결과 확인

In [ ]:
print("📊 하드웨어 뉴스 DB 현황")
print(fetch_df("""
    SELECT category, sentiment_label, COUNT(*) cnt
    FROM news_articles WHERE category LIKE 'hardware%'
    GROUP BY category, sentiment_label ORDER BY category, cnt DESC
""").to_string(index=False))

print("\n키워드 카테고리 분포:")
print(fetch_df("""
    SELECT keyword_category, COUNT(*) cnt
    FROM news_articles WHERE category LIKE 'hardware%'
    GROUP BY keyword_category ORDER BY cnt DESC
""").to_string(index=False))